# 13주차 실습: 실험 기록·모델 버전·배포 판단

강의: [PDF](../pdf/week13_mlops_pipeline.pdf) · **PDF 실제 페이지 4–16, 19–27** (인쇄 번호와 다를 수 있음)

## 학습 목표
모델과 데이터 버전을 기록하고 모의 배포 게이트·드리프트 경보를 설계한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 실험 기록과 모델 산출물

In [ ]:
import hashlib, pickle, uuid, platform
import sklearn
from labkit.common import ctr_pipeline, split_ctr
from sklearn.metrics import roc_auc_score
df=pd.read_csv(DATA/'ctr.csv'); train,valid,test=split_ctr(df)
model=ctr_pipeline().fit(train,train.clicked)
out=output_dir('week13')/uuid.uuid4().hex; out.mkdir()
auc=roc_auc_score(valid.clicked,model.predict_proba(valid)[:,1])
model_path=out/'model.pkl'
model_path.write_bytes(pickle.dumps(model))
record={'validation_auc':auc,'seed':42,'python':platform.python_version(),'sklearn':sklearn.__version__,
 'data_sha256':hashlib.sha256((DATA/'ctr.csv').read_bytes()).hexdigest(),
 'model_sha256':hashlib.sha256(model_path.read_bytes()).hexdigest(),
 'state':'candidate','model_file':str(model_path)}
save_json(out/'run.json',record)
restored=pickle.loads(model_path.read_bytes())  # 이 실습에서 직접 만든 파일만 로드
assert np.allclose(model.predict_proba(valid),restored.predict_proba(valid))
display(record)

## 드리프트·승인·롤백 모의실험

In [ ]:
from scipy.stats import ks_2samp
shifted=valid.copy(); shifted['ad_quality']=(shifted.ad_quality+.3).clip(0,1)
ks=ks_2samp(train.ad_quality,shifted.ad_quality)
quality_pass=auc>=.65  # 수업용 예시 기준, 서비스별 기준을 별도 정의
drift_alert=ks.statistic>.2
human_approved=False
decision='promote' if quality_pass and not drift_alert and human_approved else 'hold'
save_json(out/'deployment_decision.json',{'quality_pass':quality_pass,'ks_statistic':ks.statistic,
 'drift_alert':bool(drift_alert),'human_approved':human_approved,'decision':decision,'rollback_to':'previous_champion'})
assert decision=='hold'
print('모의 배포 결정:',decision,'/ 입력 드리프트는 성능 저하 자체의 증거는 아닙니다.')

## 확장 과제 및 제출
수집→검증→학습→등록→승인→배포→모니터링 DAG를 작성하세요. MLflow로 run.json의 파라미터·지표·모델을 옮기고 Canary 비율, p95 지연·오류율·성능 경보 및 롤백 기준을 설계하세요. 기본 실습은 로컬 모의 실행이며 실제 서비스 배포가 아닙니다.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.